# Notebook 1 — Simple Laptop & Server Supply Chain Data

Fictional training example. The four items and suppliers are not real records.
We will create three small tables: `item_master`, `inventory`, and `supplier`.


In [0]:
%python
catalog_name = "main"  # Change if your workspace uses another catalog
schema_name = "scm_e2e_rag_lab"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}") 

 

## A. Item Master — What is the item?

In [0]:

item_rows = [
    ("RAM001", "16 GB Laptop RAM", "Memory", "16 GB DDR4 memory module for laptops"),
    ("RAM002", "32 GB Server RAM", "Memory", "32 GB DDR5 ECC memory module for servers"),
    ("SSD001", "1 TB NVMe SSD", "Storage", "1 TB M.2 NVMe solid state drive for computers and laptops"),
    ("BAT001", "Laptop Battery", "Battery", "Rechargeable battery pack for a laptop"),
]
item_columns = ["item_id", "item_name", "category", "description"]
item_df = spark.createDataFrame(item_rows, item_columns)
item_table = f"{catalog_name}.{schema_name}.item_master"
item_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(item_table)
display(spark.table(item_table))


## B. Inventory — How many items are available, and where?

In [0]:


inventory_rows = [
    ("RAM001", "Chennai", 100),
    ("RAM002", "Chennai", 20),
    ("SSD001", "Chennai", 50),
    ("BAT001", "Bengaluru", 75),
]
inventory_columns = ["item_id", "warehouse", "available_quantity"]
inventory_df = spark.createDataFrame(inventory_rows, inventory_columns)
inventory_table = f"{catalog_name}.{schema_name}.inventory"
inventory_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(inventory_table)
display(spark.table(inventory_table))


## C. Supplier — Who supplies the item, where are they located, and how long does delivery take?

In [0]:

supplier_rows = [
    ("RAM001", "ABC Components", "Chennai", "India", 5),
    ("RAM002", "XYZ Electronics", "Taipei", "Taiwan", 12),
    ("SSD001", "ABC Components", "Chennai", "India", 7),
    ("BAT001", "PowerTech", "Penang", "Malaysia", 10),
]
supplier_columns = ["item_id", "supplier_name", "supplier_city", "supplier_country", "lead_time_days"]
supplier_df = spark.createDataFrame(supplier_rows, supplier_columns)
supplier_table = f"{catalog_name}.{schema_name}.supplier"
supplier_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(supplier_table)
display(spark.table(supplier_table))


## Checkpoint

You should now have three tables. Supplier location means where the supplier is based; warehouse means where our stock is stored. Run this query to see them joined together.

In [0]:


display(spark.sql(f"""
SELECT i.item_id, i.item_name, v.warehouse, v.available_quantity,
       s.supplier_name, s.supplier_city, s.supplier_country, s.lead_time_days
FROM {item_table} i
LEFT JOIN {inventory_table} v ON i.item_id = v.item_id
LEFT JOIN {supplier_table} s ON i.item_id = s.item_id
ORDER BY i.item_id
"""))
